# Import Libraries 

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import configparser

# DB credentials from Config.ini 

In [ ]:
config = configparser.ConfigParser()
config.read("config.ini")
db = config["database"]
db_name = db.get("dbname")
db_user = db.get("user")
db_host = db.get("host")
db_port = db.get("port")
db_password = db.get("password")
db_url = f"mysql+pymysql://{db_user}:@{db_host}:{db_port}/{db_name}"

# Create DB Engine (pymysql driver)

In [ ]:
engine = create_engine(db_url)

# Source Folder Path from config.ini

In [ ]:
fd = config["Paths"]
folder = fd.get("folder")

# Table → File Mapping Dictionary 

In [ ]:
#        Key   = target staging table name in MySQL
#        Value = full path to the source CSV file
table_file_dict = {
    "stg_transactions"  : folder + "transactions.csv",
    "stg_accounts"     : folder + "accounts.csv",
    "stg_payments"     : folder + "payments.csv",
    "stg_creditcard"   : folder + "creditcard.csv",
    "stg_loans"        : folder + "loans.csv",
    "stg_cust_profile" : folder + "cust.csv",
    "stg_branches"     : folder + "branches.csv",
    "stg_employees"    : folder + "employee.csv",
}

# Loop & Load 

In [ ]:

#        Iterates 8 times (one per dictionary entry)
#        Each iteration: reads CSV → loads to MySQL staging table
for table, file in table_file_dict.items():
    # Read CSV into a pandas DataFrame (in-memory table)
    df = pd.read_csv(file)

    # Load DataFrame → MySQL staging table (replace if already exists)
    df.to_sql(table, con=engine, index=False, if_exists="replace")

    print(f"Rows loaded into table: {table}")

In [ ]:
# transactions = pd.read_csv(folder +f'/transactions.csv')
# accounts = pd.read_csv(folder +f'/accounts.csv')
# payments = pd.read_csv(folder +f'/payments.csv')
# creditcard = pd.read_csv(folder +f'/creditcard.csv')
# loans = pd.read_csv(folder +f'/loans.csv')
# cust = pd.read_csv(folder +f'/cust.csv')
# branches = pd.read_csv(folder +f'/branches.csv')
# employee = pd.read_csv(folder +f'/employee.csv')


In [ ]:
loans.to_sql('stg_loans', con=engine, index=False, if_exists='replace')

In [ ]:
query = "SELECT * FROM stg_loans "
df_stg_loans = pd.read_sql(query, con=engine)
df_stg_loans

In [ ]:
df_stg_loans['HighValueFlag'] = np.where(df_stg_loans['Amount'] >= 100000, 'Y', 'N')
df_stg_loans.to_sql('fact_loans', con=engine, index=False, if_exists='replace')

In [ ]:
query = "SELECT * FROM fact_loans "
df_fact_loans = pd.read_sql(query, con=engine)
df_fact_loans

In [ ]:
#df_stg_loans = df_stg_loans.drop(columns=['HighValueFlag'], inplace=True)
#df_stg_loans.to_sql('stg_loans', con=engine, index=False, if_exists='replace')

In [ ]:
df_stg_loans['DateDiff'] = (pd.to_datetime(df_stg_loans['EndDate']) - pd.to_datetime(df_stg_loans['StartDate'])).dt.days
df_stg_loans

In [ ]:
df_stg_loans.to_sql('fact_loans', con=engine, index=False, if_exists='replace')

In [ ]:
df_stg_loans

In [ ]:
df_fact_loans.shape[0]